In [7]:
import xarray as xr
import numpy as np
import pandas as pd
import os

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature


from plot_utils_era5 import *

In [8]:

def data_loader(date, time=None, vars=None, lat_max_min=None, lon_min_max=None, datafolder_path='../data'):

    file_name = os.path.join(datafolder_path, date + '.nc')
    ds = xr.open_dataset(file_name)

    # Convert all longitudes to [-180, 180] format
    ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180))
    
    # Apply longitude range condition
    if lon_min_max is not None:
        l0, l1 = lon_min_max
        if l0 < l1:
            ds = ds.where((ds.longitude >= l0) & (ds.longitude <= l1), drop=True)
        else:  # Handle wrapping across -180/180
            ds = ds.where((ds.longitude >= l0) | (ds.longitude <= l1), drop=True)

    # Latitude selection
    if lat_max_min is not None:
        lat0, lat1 = sorted(lat_max_min)
        ds = ds.where((ds.latitude >= lat0) & (ds.latitude <= lat1), drop=True)
    

    # Time selection
    if time is not None and 'time' in ds:
        ds = ds.sel(time=time)

    # Variable selection
    if vars is not None:
        ds = ds[vars]

    # Convert to DataFrame
    df = ds.to_dataframe().reset_index()
    df = df.rename(columns={"longitude": "lon", "latitude": "lat"})
    df = df.drop(columns=[c for c in ['valid_time', 'pressure_level', 'number', 'expver'] if c in df.columns], errors='ignore')

    return df


def plot_era5(
    df,
    var,
    *,
    projection=ccrs.PlateCarree(),
    cmap="viridis",
    marker_size=10,
    vmin=None,
    vmax=None,
    title=None,
):
    """
    Plot a global variable from a pandas DataFrame with lat/lon columns.

    Parameters
    ----------
    df : pandas.DataFrame
        Must contain columns 'lat', 'lon', and `var`
    var : str
        Column name of the variable to plot
    projection : cartopy.crs
        Map projection for display
    cmap : str
        Matplotlib colormap
    marker_size : float
        Size of scatter markers
    vmin, vmax : float, optional
        Color limits
    title : str, optional
        Plot title
    """

    if "lat" not in df.columns or "lon" not in df.columns:
        raise ValueError("DataFrame must contain 'lat' and 'lon' columns")

    if var not in df.columns:
        raise ValueError(f"Column '{var}' not found in DataFrame")

    fig = plt.figure(figsize=(12, 6))
    ax = plt.axes(projection=projection)

    # Set global extent
    ax.set_global()

    # Add map features
    ax.add_feature(cfeature.LAND, facecolor="lightgray")
    ax.add_feature(cfeature.OCEAN, facecolor="white")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_feature(cfeature.LAKES, alpha=0.5)
    ax.add_feature(cfeature.RIVERS, linewidth=0.5)

    # Plot data
    sc = ax.scatter(
        df["lon"],
        df["lat"],
        c=df[var],
        s=marker_size,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        transform=ccrs.PlateCarree(),
    )

    #cb = plt.colorbar(sc, ax=ax, orientation="horizontal", pad=0.05)
    #cb.set_label(var)

    if title is None:
        title = f"Global map of {var}"
    ax.set_title(title)

    plt.show()


In [9]:
def plot_era5(
    df,
    var,
    *,
    lat_min=None,
    lat_max=None,
    lon_min=None,
    lon_max=None,
    projection=ccrs.PlateCarree(),
    cmap="viridis",
    marker_size=10,
    vmin=None,
    vmax=None,
    title=None,
):
    """
    Plot ERA5 data from a pandas DataFrame with optional lat/lon bounds.
    """

    if "lat" not in df.columns or "lon" not in df.columns:
        raise ValueError("DataFrame must contain 'lat' and 'lon' columns")

    if var not in df.columns:
        raise ValueError(f"Column '{var}' not found in DataFrame")

    # Apply geographic filtering
    plot_df = df

    if lat_min is not None:
        plot_df = plot_df.loc[plot_df.lat >= lat_min]
    if lat_max is not None:
        plot_df = plot_df.loc[plot_df.lat <= lat_max]
    if lon_min is not None:
        plot_df = plot_df.loc[plot_df.lon >= lon_min]
    if lon_max is not None:
        plot_df = plot_df.loc[plot_df.lon <= lon_max]

    fig = plt.figure(figsize=(12, 6))
    ax = plt.axes(projection=projection)

    # Set extent if bounds provided
    if None not in (lon_min, lon_max, lat_min, lat_max):
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    else:
        ax.set_global()

    # Map features
    ax.add_feature(cfeature.LAND, facecolor="lightgray")
    ax.add_feature(cfeature.OCEAN, facecolor="white")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_feature(cfeature.LAKES, alpha=0.5)
    ax.add_feature(cfeature.RIVERS, linewidth=0.5)

    # Plot
    sc = ax.scatter(
        plot_df["lon"],
        plot_df["lat"],
        c=plot_df[var],
        s=marker_size,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        transform=ccrs.PlateCarree(),
    )

    if title is None:
        title = f"Map of {var}"
    ax.set_title(title)

    plt.show()

In [11]:
df = data_loader('2018-01-15')
#df = df.dropna()
print(df.columns)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/siche9897/Github/Weather-Forecasting/nofe-basicfiles/ERA5/data/2018-01-15.nc'

In [ ]:
plot_era5(df, 'd')

In [ ]:
for v in df.columns:
    print(v)
    plot_era5(df, v, lat_min=30, lat_max=70, lon_min=-10, lon_max=30)

In [ ]:
import cdsapi

dataset = "reanalysis-era5-single-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": [
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "2m_dewpoint_temperature",
        "2m_temperature",
        "mean_sea_level_pressure",
        "mean_wave_direction",
        "mean_wave_period",
        "sea_surface_temperature",
        "significant_height_of_combined_wind_waves_and_swell",
        "surface_pressure",
        "total_precipitation"
    ],
    "year": ["2025"],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "01:00", "02:00",
        "03:00", "04:00", "05:00",
        "06:00", "07:00", "08:00",
        "09:00", "10:00", "11:00",
        "12:00", "13:00", "14:00",
        "15:00", "16:00", "17:00",
        "18:00", "19:00", "20:00",
        "21:00", "22:00", "23:00"
    ],
    "data_format": "grib",
    "download_format": "zip"
}

client = cdsapi.Client()
client.retrieve(dataset, request).download('era5_2025_allvars.zip')
